In [3]:
import tensorflow as tf
print(tf.keras.__version__)

3.12.1


In [1]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,     
    width_shift_range=0.2,  
    height_shift_range=0.2, 
    horizontal_flip=True,   
    fill_mode='nearest',
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    'nsfw_dataset_v1',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = val_datagen.flow_from_directory(
    'nsfw_dataset_v1',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 22400 images belonging to 5 classes.
Found 5600 images belonging to 5 classes.


In [2]:
print(train_generator.class_indices)

{'drawings': 0, 'hentai': 1, 'neutral': 2, 'porn': 3, 'sexy': 4}


In [3]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dense(5, activation='softmax')
])

model.compile(
    optimizer=Adam(),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [4]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10, 
    callbacks=[early_stop]
)

model.save("nsfw_mobilenet_model.keras")

Epoch 1/10
700/700 ━━━━━━━━━━━━━━━━━━━━ 1769s 3s/step - accuracy: 0.7847 - loss: 0.5552 - val_accuracy: 0.8182 - val_loss: 0.4856
Epoch 2/10
700/700 ━━━━━━━━━━━━━━━━━━━━ 1083s 2s/step - accuracy: 0.8229 - loss: 0.4598 - val_accuracy: 0.8255 - val_loss: 0.4599
Epoch 3/10
700/700 ━━━━━━━━━━━━━━━━━━━━ 1195s 2s/step - accuracy: 0.8343 - loss: 0.4308 - val_accuracy: 0.8229 - val_loss: 0.4567
Epoch 4/10
700/700 ━━━━━━━━━━━━━━━━━━━━ 1087s 2s/step - accuracy: 0.8421 - loss: 0.4079 - val_accuracy: 0.8304 - val_loss: 0.4301
Epoch 5/10
700/700 ━━━━━━━━━━━━━━━━━━━━ 1247s 2s/step - accuracy: 0.8515 - loss: 0.3875 - val_accuracy: 0.8416 - val_loss: 0.4170
Epoch 6/10
700/700 ━━━━━━━━━━━━━━━━━━━━ 1499s 2s/step - accuracy: 0.8517 - loss: 0.3797 - val_accuracy: 0.8146 - val_loss: 0.5013
Epoch 7/10
700/700 ━━━━━━━━━━━━━━━━━━━━ 1910s 3s/step - accuracy: 0.8591 - loss: 0.3657 - val_accuracy: 0.8400 - val_loss: 0.4258
